## TRAINING DATA

In [29]:
import os
import cv2
import numpy as np
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

In [9]:
# Define your dataset paths
TRAIN_DIR = r'C:\Users\igles\OneDrive\Documents\MOR-TEAM-13\TRAINING FOLDER\Banana picker.v5i.folder\train\Lakatan'      # Should contain ONLY normal images
VALID_DIR = r'C:\Users\igles\OneDrive\Documents\MOR-TEAM-13\TRAINING FOLDER\Banana picker.v5i.folder\valid\Lakatan'      # Should contain mixed normal and anomaly
TEST_DIR  = r'C:\Users\igles\OneDrive\Documents\MOR-TEAM-13\TRAINING FOLDER\Banana picker.v5i.folder\test\Lakatan'       # Should contain mixed normal and anomaly

# Image settings
IMG_SIZE = (64, 64)  # Resize images to this size to reduce processing time

In [15]:
# --- UPDATED LOAD FUNCTION ---
def load_images_from_folder(folder):
    images = []
    filenames = []
    
    # Simply list all files in the folder
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        
        # Check if it is an image file (basic check)
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                # Read in grayscale
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) 
                if img is not None:
                    img = cv2.resize(img, IMG_SIZE)
                    images.append(img.flatten())
                    filenames.append(filename)
            except Exception as e:
                print(f"Error loading {filename}: {e}")
                
    return np.array(images), filenames

In [22]:
print("Loading Training Data...")
X_train, _ = load_images_from_folder(TRAIN_DIR)

print("Loading Validation Data...")
X_valid, valid_files = load_images_from_folder(VALID_DIR)

print("Loading Test Data...")
X_test, test_files = load_images_from_folder(TEST_DIR)

Loading Training Data...
Loading Validation Data...
Loading Test Data...


In [23]:
print(f"Train Shape: {X_train.shape}")
print(f"Valid Shape: {X_valid.shape}")
print(f"Test Shape: {X_test.shape}")


Train Shape: (280, 4096)
Valid Shape: (80, 4096)
Test Shape: (40, 4096)


In [24]:
# --- PREPROCESSING ---
# Normalize to 0-1 range
X_train = X_train / 255.0
X_valid = X_valid / 255.0
X_test = X_test / 255.0

In [25]:
# --- TRAIN MODEL ---
# nu: The upper bound on the fraction of training errors (outliers).
# If you set nu=0.05, the model allows roughly 5% of training data to be outside the boundary.
model = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)

In [26]:
print("Training One-Class SVM...")
model.fit(X_train)
print("Training Complete.")

Training One-Class SVM...
Training Complete.


In [27]:
# --- EVALUATE ON VALIDATION ---
# The model predicts 1 for Inlier (Normal) and -1 for Outlier (Anomaly)
print("\n--- Validation Results ---")
y_pred_valid = model.predict(X_valid)

# Calculate statistics
n_normal = np.sum(y_pred_valid == 1)
n_outliers = np.sum(y_pred_valid == -1)
total = len(y_pred_valid)



--- Validation Results ---


In [28]:
print(f"Total Images: {total}")
print(f"Predicted Normal (1):  {n_normal} ({100*n_normal/total:.2f}%)")
print(f"Predicted Anomaly (-1): {n_outliers} ({100*n_outliers/total:.2f}%)")

print("\nNote: Since your validation set only contains 'Normal' images,")
print("the 'Predicted Anomaly' count represents False Positives (Good data flagged as Bad).")
print("If this number is high, try lowering the 'nu' parameter or checking your data quality.")

# --- EVALUATE ON TEST ---
print("\n--- Test Results ---")
y_pred_test = model.predict(X_test)

n_normal_test = np.sum(y_pred_test == 1)
n_outliers_test = np.sum(y_pred_test == -1)
total_test = len(y_pred_test)

print(f"Total Images: {total_test}")
print(f"Predicted Normal (1):  {n_normal_test} ({100*n_normal_test/total_test:.2f}%)")
print(f"Predicted Anomaly (-1): {n_outliers_test} ({100*n_outliers_test/total_test:.2f}%)")

Total Images: 80
Predicted Normal (1):  61 (76.25%)
Predicted Anomaly (-1): 19 (23.75%)

Note: Since your validation set only contains 'Normal' images,
the 'Predicted Anomaly' count represents False Positives (Good data flagged as Bad).
If this number is high, try lowering the 'nu' parameter or checking your data quality.

--- Test Results ---
Total Images: 40
Predicted Normal (1):  26 (65.00%)
Predicted Anomaly (-1): 14 (35.00%)


In [31]:
print(  classification_report(y_true_valid, y_pred_valid) )

              precision    recall  f1-score   support

        -1.0       0.00      0.00      0.00         0
         1.0       1.00      0.76      0.87        80

    accuracy                           0.76        80
   macro avg       0.50      0.38      0.43        80
weighted avg       1.00      0.76      0.87        80



c:\Users\igles\OneDrive\Documents\MOR-TEAM-13\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\igles\OneDrive\Documents\MOR-TEAM-13\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\igles\OneDrive\Documents\MOR-TEAM-13\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

# TEST 2

In [32]:
from sklearn.svm import OneClassSVM
import numpy as np

# 1. Define the grid of parameters to test
# nu: Controls the trade-off. Lower = stricter (less anomalies allowed). Higher = looser.
# gamma: Defines how far the influence of a single training example reaches. 
#        'scale' is default (1 / (n_features * X.var()))
#        Lower gamma = smoother decision boundary.
param_grid = {
    'nu': [0.01, 0.05, 0.1, 0.2, 0.3], 
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

best_score = -1
best_params = {}
best_model = None

print(f"Starting Optimization with {len(param_grid['nu']) * len(param_grid['gamma'])} combinations...")

# 2. Loop through every combination
for nu in param_grid['nu']:
    for gamma in param_grid['gamma']:
        
        # Train model
        clf = OneClassSVM(kernel='rbf', nu=nu, gamma=gamma)
        clf.fit(X_train)
        
        # Predict on Validation
        y_pred = clf.predict(X_valid)
        
        # Calculate "Accuracy" for Normal Data
        # (How many validation images were correctly identified as Normal/1?)
        current_accuracy = np.mean(y_pred == 1)
        
        # Print progress
        print(f"nu={nu}, gamma={gamma} --> Validation Accuracy: {current_accuracy:.4f}")
        
        # Save the best model
        if current_accuracy > best_score:
            best_score = current_accuracy
            best_params = {'nu': nu, 'gamma': gamma}
            best_model = clf

# 3. Report Results
print("\n" + "="*30)
print("OPTIMIZATION COMPLETE")
print("="*30)
print(f"Best Validation Accuracy: {best_score:.4f}")
print(f"Best Parameters: {best_params}")
print("="*30 + "\n")

Starting Optimization with 30 combinations...
nu=0.01, gamma=scale --> Validation Accuracy: 0.7625
nu=0.01, gamma=auto --> Validation Accuracy: 0.9500
nu=0.01, gamma=0.001 --> Validation Accuracy: 0.9250
nu=0.01, gamma=0.01 --> Validation Accuracy: 0.7125
nu=0.01, gamma=0.1 --> Validation Accuracy: 0.0000
nu=0.01, gamma=1 --> Validation Accuracy: 0.0000
nu=0.05, gamma=scale --> Validation Accuracy: 0.7625
nu=0.05, gamma=auto --> Validation Accuracy: 0.9375
nu=0.05, gamma=0.001 --> Validation Accuracy: 0.9250
nu=0.05, gamma=0.01 --> Validation Accuracy: 0.7125
nu=0.05, gamma=0.1 --> Validation Accuracy: 0.0000
nu=0.05, gamma=1 --> Validation Accuracy: 0.0000
nu=0.1, gamma=scale --> Validation Accuracy: 0.7500
nu=0.1, gamma=auto --> Validation Accuracy: 0.9000
nu=0.1, gamma=0.001 --> Validation Accuracy: 0.8875
nu=0.1, gamma=0.01 --> Validation Accuracy: 0.7125
nu=0.1, gamma=0.1 --> Validation Accuracy: 0.0000
nu=0.1, gamma=1 --> Validation Accuracy: 0.0000
nu=0.2, gamma=scale --> Valida

In [33]:
# Use the optimized model for final evaluation
y_pred_final = best_model.predict(X_valid)

# Generate labels (since all are normal)
y_true_valid = np.ones(len(y_pred_final))

from sklearn.metrics import classification_report
print("Final Classification Report (Optimized):")
print(classification_report(y_true_valid, y_pred_final, target_names=['Anomaly', 'Normal'], labels=[-1, 1], zero_division=0))

Final Classification Report (Optimized):
              precision    recall  f1-score   support

     Anomaly       0.00      0.00      0.00         0
      Normal       1.00      0.95      0.97        80

    accuracy                           0.95        80
   macro avg       0.50      0.47      0.49        80
weighted avg       1.00      0.95      0.97        80



In [34]:
def test_single_image(image_path, model, target_size=(64, 64)):
    """
    Loads a single image, processes it, and predicts if it is Normal or Anomaly.
    """
    # 1. Load the image (Grayscale to match training)
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        return "Error: Could not load image. Check path."

    # 2. Resize to match training size (e.g., 64x64)
    img_resized = cv2.resize(img, target_size)

    # 3. Flatten and Normalize
    # Flatten: (64, 64) -> (4096,)
    # Reshape: (4096,) -> (1, 4096) because sklearn expects a list of samples
    img_flat = img_resized.flatten().reshape(1, -1)
    img_normalized = img_flat / 255.0

    # 4. Predict
    prediction = model.predict(img_normalized) # Returns [1] or [-1]

    # 5. Return Label and Image for visualization
    label = "Normal" if prediction[0] == 1 else "Anomaly"
    return label, img_resized

In [35]:
newimg= r'C:\Users\igles\OneDrive\Documents\MOR-TEAM-13\TRAINING FOLDER\Banana picker.v5i.folder\train\Lakatan\20240626_094358_jpg.rf.8dd8f9efe11e35dfb84a88f73ed807d4.jpg'
test_single_image(newimg, best_model, target_size=IMG_SIZE)

('Normal',
 array([[35, 36, 34, ..., 56, 57, 55],
        [33, 35, 34, ..., 59, 56, 55],
        [34, 36, 34, ..., 62, 53, 55],
        ...,
        [25, 24, 26, ..., 51, 54, 51],
        [25, 24, 25, ..., 52, 52, 52],
        [25, 25, 26, ..., 51, 52, 52]], shape=(64, 64), dtype=uint8))

In [38]:
newimg= r'C:\Users\igles\Downloads\test 3.JPG'
test_single_image(newimg, best_model, target_size=IMG_SIZE)

('Anomaly',
 array([[178, 186, 184, ..., 216, 215, 213],
        [183, 185, 187, ..., 217, 216, 214],
        [188, 189, 187, ..., 219, 217, 215],
        ...,
        [153, 157, 158, ..., 213, 212, 211],
        [155, 157, 161, ..., 213, 212, 209],
        [155, 159, 160, ..., 210, 213, 210]], shape=(64, 64), dtype=uint8))

In [39]:
newimg= r"C:\Users\igles\Downloads\test4.jpg"
test_single_image(newimg, best_model, target_size=IMG_SIZE)

('Anomaly',
 array([[202, 202, 203, ..., 215, 215, 215],
        [204, 204, 205, ..., 217, 217, 217],
        [204, 205, 207, ..., 218, 218, 218],
        ...,
        [ 96,  92, 102, ..., 160, 162, 163],
        [ 92,  97,  98, ..., 161, 161, 161],
        [104, 100, 108, ..., 163, 158, 153]], shape=(64, 64), dtype=uint8))